# 0교시: Day 4 복습 실습

## 실습 목표
- SELECT, WHERE, GROUP BY, HAVING, JOIN 복습
- 배운 문법을 조합하여 비즈니스 질문 해결

---

## 실습 안내

- 총 3문제 (난이도: 중)
- 각 문제 해결 후 힌트와 정답 확인
- 배운 범위: SELECT, WHERE, ORDER BY, 집계함수, GROUP BY, HAVING, JOIN

---

## 문제 1: 고객 지역별 주문 통계

**요청 사항**

고객이 거주하는 지역별로 **완료된 주문**의 통계를 분석하세요.

필요한 정보:
- 지역
- 주문 건수
- 총 매출
- 평균 주문금액 (정수로 반올림)

정렬: 총 매출이 높은 순서대로

---

<details>
<summary>힌트 1: 어떤 테이블을 사용해야 할까요?</summary>

- `users` 테이블: 고객의 `region`(지역) 정보가 있음
- `orders` 테이블: `total_amount`, `status` 정보가 있음
- → 두 테이블을 `user_id`로 JOIN 해야 함
</details>

<details>
<summary>힌트 2: 쿼리 구조</summary>

```sql
SELECT
    u.region as 지역,
    ___(*) as 주문수,
    ___(o.total_amount) as 총매출,
    ROUND(___(o.total_amount)) as 평균금액
FROM orders o
JOIN users u ON o.___ = u.___
WHERE o.status = '___'
GROUP BY ___
ORDER BY ___ DESC;
```
</details>

<details>
<summary>정답</summary>

```sql
SELECT
    u.region as 지역,
    COUNT(*) as 주문수,
    SUM(o.total_amount) as 총매출,
    ROUND(AVG(o.total_amount)) as 평균금액
FROM orders o
JOIN users u ON o.user_id = u.user_id
WHERE o.status = 'completed'
GROUP BY u.region
ORDER BY 총매출 DESC;
```

결과 예시:
```
  지역  | 주문수 | 총매출  | 평균금액
--------+--------+---------+----------
 성북   |     43 | 1281994 |    29814
 노원   |     43 | 1245523 |    28966
 강서   |     41 | 1074933 |    26218
 ...
```
</details>

---
## 문제 2: 인기 식당 매출 랭킹

**요청 사항**

완료된 주문이 **20건 이상**인 식당만 골라서 매출 분석을 하세요.

필요한 정보:
- 식당명
- 카테고리
- 주문 건수
- 총 매출

정렬: 총 매출이 높은 순서대로

---

<details>
<summary>힌트 1: 사용할 문법</summary>

- `JOIN`: orders와 restaurants 연결
- `WHERE`: 완료된 주문만 필터
- `GROUP BY`: 식당별로 그룹화
- `HAVING`: 그룹화 후 20건 이상 조건
</details>

<details>
<summary>힌트 2: WHERE vs HAVING</summary>

- `WHERE o.status = 'completed'`: 그룹화 **전**에 개별 행 필터
- `HAVING COUNT(*) >= 20`: 그룹화 **후**에 그룹 필터

```sql
SELECT ...
FROM orders o
JOIN restaurants r ON ...
WHERE o.status = 'completed'  -- 먼저 필터
GROUP BY ...
HAVING COUNT(*) >= 20         -- 그룹 필터
ORDER BY ...;
```
</details>

<details>
<summary>정답</summary>

```sql
SELECT
    r.name as 식당명,
    r.category as 카테고리,
    COUNT(*) as 주문수,
    SUM(o.total_amount) as 총매출
FROM orders o
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
WHERE o.status = 'completed'
GROUP BY r.restaurant_id, r.name, r.category
HAVING COUNT(*) >= 20
ORDER BY 총매출 DESC;
```

결과 예시:
```
   식당명   | 카테고리 | 주문수 | 총매출
------------+----------+--------+--------
 된장마을   | 한식     |     27 | 760934
 굽네치킨   | 치킨     |     26 | 720774
 도미노피자 | 피자     |     22 | 667411
 피자나라   | 피자     |     20 | 534067
```
</details>

---
## 문제 3: 12월 치킨 주문 지역별 분석

**요청 사항**

2025년 12월에 **치킨** 카테고리에서 **완료된** 주문을 지역별로 분석하세요.

필요한 정보:
- 지역
- 치킨 주문 건수
- 치킨 매출

정렬: 치킨 매출이 높은 순서대로

---

<details>
<summary>힌트 1: 필요한 테이블</summary>

- `orders`: 주문 정보 (금액, 날짜, 상태)
- `users`: 지역 정보
- `restaurants`: 카테고리 정보
- → 3개 테이블 JOIN 필요!
</details>

<details>
<summary>힌트 2: WHERE 조건</summary>

3가지 조건을 AND로 연결:
- `r.category = '치킨'`
- `o.status = 'completed'`
- `o.created_at >= '2025-12-01' AND o.created_at < '2026-01-01'`
</details>

<details>
<summary>힌트 3: 3테이블 JOIN 구조</summary>

```sql
FROM orders o
JOIN users u ON o.user_id = u.user_id
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
```
</details>

<details>
<summary>정답</summary>

```sql
SELECT
    u.region as 지역,
    COUNT(*) as 치킨주문수,
    SUM(o.total_amount) as 치킨매출
FROM orders o
JOIN users u ON o.user_id = u.user_id
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
WHERE r.category = '치킨'
  AND o.status = 'completed'
  AND o.created_at >= '2025-12-01'
  AND o.created_at < '2026-01-01'
GROUP BY u.region
ORDER BY 치킨매출 DESC;
```

결과 예시:
```
  지역  | 치킨주문수 | 치킨매출
--------+------------+----------
 노원   |          4 |   135278
 성북   |          4 |   101418
 종로   |          3 |    88428
 강남   |          2 |    71263
 ...
```
</details>

---
## 복습 포인트

| 문제 | 핵심 문법 |
|------|----------|
| 문제 1 | 2테이블 JOIN + GROUP BY + 집계함수 |
| 문제 2 | JOIN + WHERE + GROUP BY + HAVING |
| 문제 3 | 3테이블 JOIN + 복합 WHERE 조건 |

### 기억할 것
1. **JOIN ON 조건**: 연결할 컬럼 명확히 지정
2. **WHERE vs HAVING**: 개별 행 vs 그룹 조건
3. **실행 순서**: FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY

---

---


# 1교시: 서브쿼리

## 학습 목표
- 서브쿼리가 무엇인지, 왜 필요한지 이해
- WHERE절 스칼라 서브쿼리 작성
- IN 서브쿼리 작성
- CTE(WITH) 기본 사용법

---

## 1. 서브쿼리란?

**쿼리 안에 포함된 또 다른 쿼리**

```
SELECT *
FROM orders
WHERE total_amount > (SELECT AVG(total_amount) FROM orders);
                      ┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛
                                  서브쿼리
```

### 왜 필요할까?

**상황**: "평균 주문금액보다 높은 주문을 찾고 싶다"

| 접근 방식 | 문제점 |
|-----------|--------|
| 평균 먼저 조회 → 값 복사해서 WHERE에 입력 | 데이터 바뀌면 다시 해야 함 |
| 서브쿼리로 한 번에 처리 | 항상 최신 평균값 사용 ✅ |

> **핵심**: 서브쿼리는 **동적으로 조건을 계산**할 때 사용

---
## 2. 스칼라 서브쿼리 (단일 값 반환)

### 평균보다 비싼 주문 찾기

**Step 1: 평균 먼저 확인**
```sql
SELECT AVG(total_amount) FROM orders;
-- 결과: 약 25,700
```

**Step 2: 하드코딩으로 시도 (❌ 나쁜 방법)**
```sql
SELECT * FROM orders
WHERE total_amount > 25700
LIMIT 10;
-- 문제: 데이터가 추가되면 평균이 바뀌는데?
```

**Step 3: 서브쿼리로 해결 (✅ 좋은 방법)**
```sql
SELECT * FROM orders
WHERE total_amount > (SELECT AVG(total_amount) FROM orders)
LIMIT 10;
```

**실행 순서**:
1. `(SELECT AVG(total_amount) FROM orders)` 먼저 실행 → 25700
2. `WHERE total_amount > 25700` 으로 필터링

### 🚨 의도적 오류: 서브쿼리가 여러 행 반환

```sql
-- 모든 주문금액과 비교하려고 시도
SELECT * FROM orders
WHERE total_amount > (SELECT total_amount FROM orders);
```

**에러 메시지:**
```
ERROR: more than one row returned by a subquery used as an expression
```

**원인**: 스칼라 서브쿼리는 **반드시 1개 값만** 반환해야 함

**해결 방법**:
- 집계 함수 사용: `AVG()`, `MAX()`, `MIN()` 등
- `LIMIT 1` 추가
- 여러 값이면 `IN` 서브쿼리 사용

### 또 다른 예시: 가장 최근 주문일의 주문 조회

```sql
-- 가장 최근 주문일 확인
SELECT MAX(created_at) FROM orders;
-- 결과: 2025-12-31 ...

-- 그 날의 주문만 조회
SELECT * FROM orders
WHERE DATE(created_at) = (
    SELECT DATE(MAX(created_at)) FROM orders
);
```

> **실무 활용**: ETL에서 "오늘 들어온 데이터만 추출" 할 때 자주 사용

---
## 3. IN 서브쿼리 (여러 값 반환)

### 강남 지역 고객의 주문만 조회

**Step 1: 강남 고객 ID 목록 확인**
```sql
SELECT user_id FROM users WHERE region = '강남';
-- 결과: 1, 11, 21, 31, ... (여러 개)
```

**Step 2: IN 서브쿼리로 조회**
```sql
SELECT * FROM orders
WHERE user_id IN (
    SELECT user_id FROM users WHERE region = '강남'
)
LIMIT 10;
```

> **IN 서브쿼리**: 서브쿼리가 반환한 **여러 값 중 하나와 일치**하면 선택

### 치킨 카테고리 식당의 주문 조회

```sql
SELECT * FROM orders
WHERE restaurant_id IN (
    SELECT restaurant_id
    FROM restaurants
    WHERE category = '치킨'
)
LIMIT 10;
```

### 서브쿼리 vs JOIN - 언제 뭘 쓸까?

**같은 결과, 다른 방법**

**서브쿼리 방식:**
```sql
SELECT * FROM orders
WHERE restaurant_id IN (
    SELECT restaurant_id FROM restaurants WHERE category = '치킨'
);
```

**JOIN 방식:**
```sql
SELECT o.*
FROM orders o
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
WHERE r.category = '치킨';
```

| 상황 | 추천 방식 |
|------|-----------|
| 연결된 테이블의 **컬럼이 필요 없음** | 서브쿼리 (간단) |
| 연결된 테이블의 **컬럼도 조회** | JOIN |
| 가독성 중시 | 취향에 따라 |

> **실무 팁**: 우선 읽기 쉬운 방식으로 작성, 성능 문제 시 최적화

---
## 4. CTE (WITH) - 서브쿼리를 읽기 쉽게

### 문제: 서브쿼리가 복잡해지면?

```sql
-- 강남 지역의 2025년 가입 고객의 12월 주문
SELECT * FROM orders
WHERE user_id IN (
    SELECT user_id
    FROM users
    WHERE region = '강남'
    AND created_at >= '2025-01-01'
)
AND created_at >= '2025-12-01'
LIMIT 10;
```

→ 괄호 안이 길어지면 **읽기 어려움**

### CTE로 단계별 정리

```sql
-- CTE (Common Table Expression)
WITH gangnam_users AS (
    SELECT user_id
    FROM users
    WHERE region = '강남'
    AND created_at >= '2025-01-01'
)
SELECT * FROM orders
WHERE user_id IN (SELECT user_id FROM gangnam_users)
AND created_at >= '2025-12-01'
LIMIT 10;
```

**CTE 문법**:
```
WITH 이름 AS (
    서브쿼리
)
SELECT ... FROM ... WHERE ...
```

> **WITH** = "이 쿼리를 미리 정의해두고, 아래에서 사용할게"

### CTE의 장점

| 장점 | 설명 |
|------|------|
| **가독성** | 위에서 아래로 읽으면 됨 |
| **재사용** | 같은 CTE를 여러 번 참조 가능 |
| **디버깅** | CTE만 따로 실행해서 확인 가능 |

### 여러 CTE 정의하기

```sql
WITH
    gangnam_users AS (
        SELECT user_id FROM users WHERE region = '강남'
    ),
    chicken_restaurants AS (
        SELECT restaurant_id FROM restaurants WHERE category = '치킨'
    )
SELECT * FROM orders
WHERE user_id IN (SELECT user_id FROM gangnam_users)
AND restaurant_id IN (SELECT restaurant_id FROM chicken_restaurants)
LIMIT 10;
```

> **Spark SQL에서도 동일하게 사용!** 복잡한 ETL 로직 작성 시 CTE 필수

---
## 5. 실무에서 서브쿼리 활용

### ETL 파이프라인에서의 역할

```
┌─────────────────────────────────────────────────────────────┐
│                    ETL 파이프라인                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Extract (추출)     Transform (변환)     Load (적재)         │
│       ↓                  ↓                  ↓               │
│   서브쿼리로          JOIN, CASE로        결과 테이블에       │
│   조건 필터링         데이터 가공          INSERT            │
│                                                             │
│  "평균 이상만"        "상태값 변환"       "마트 테이블로"     │
│  "최근 7일만"         "NULL 처리"                           │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### Spark SQL에서도 똑같이!

```python
# PySpark에서 SQL 사용
spark.sql("""
    WITH active_users AS (
        SELECT user_id FROM users WHERE status = 'active'
    )
    SELECT * FROM orders
    WHERE user_id IN (SELECT user_id FROM active_users)
""")
```

> **지금 배우는 SQL이 Spark에서도 그대로 사용됩니다!**

---
## 6. 연습 문제

### 문제 1: 스칼라 서브쿼리
평균 주문금액보다 높은 주문만 조회하세요.

<details>
<summary>힌트</summary>

`WHERE total_amount > (SELECT AVG(...) FROM ...)`
</details>

<details>
<summary>정답</summary>

```sql
SELECT * FROM orders
WHERE total_amount > (SELECT AVG(total_amount) FROM orders)
LIMIT 20;
```
</details>

### 문제 2: IN 서브쿼리
한 번이라도 주문한 적 있는 고객만 조회하세요.

<details>
<summary>힌트</summary>

1. orders 테이블에서 user_id 목록 추출
2. users 테이블에서 IN으로 필터링
</details>

<details>
<summary>정답</summary>

```sql
SELECT * FROM users
WHERE user_id IN (
    SELECT DISTINCT user_id FROM orders
);
```
</details>

### 문제 3: CTE 활용
VIP 고객(5회 이상 주문)의 12월 주문을 조회하세요.

<details>
<summary>힌트</summary>

1. WITH vip_customers AS (...) 로 5회 이상 주문한 user_id 정의
2. 메인 쿼리에서 12월 조건 추가
</details>

<details>
<summary>정답</summary>

```sql
WITH vip_customers AS (
    SELECT user_id
    FROM orders
    GROUP BY user_id
    HAVING COUNT(*) >= 5
)
SELECT * FROM orders
WHERE user_id IN (SELECT user_id FROM vip_customers)
AND created_at >= '2025-12-01'
LIMIT 20;
```
</details>

---
## 7. 핵심 정리

| 서브쿼리 유형 | 위치 | 반환 값 | 예시 |
|--------------|------|---------|------|
| 스칼라 | WHERE | 단일 값 | `> (SELECT AVG(...))` |
| IN | WHERE | 여러 값 | `IN (SELECT id FROM ...)` |
| CTE | WITH절 | 임시 테이블 | `WITH t AS (...) SELECT ...` |

### 기억할 것
1. 스칼라 서브쿼리는 **반드시 1개 값만** 반환
2. 여러 값 비교는 **IN** 사용
3. 복잡한 서브쿼리는 **CTE(WITH)** 로 정리
4. **Spark SQL에서도 동일하게 사용!**

---

## 다음 시간 예고
ROW_NUMBER로 그룹별 순위 매기기, CASE WHEN과 COALESCE로 데이터 정제

---


# 2교시: 윈도우 함수와 데이터 정제

## 학습 목표
- ROW_NUMBER로 그룹별 순위 매기기
- CASE WHEN으로 조건별 값 변환
- COALESCE로 NULL 처리
- 데이터 정제가 DE 업무에서 어떤 위치인지 이해

---

## 1. ROW_NUMBER - 그룹별 순위 매기기

### GROUP BY의 한계

```sql
-- 고객별 주문 수 집계
SELECT user_id, COUNT(*) as order_count
FROM orders
GROUP BY user_id;
```

결과: **100행** (고객당 1행으로 축소)

```
user_id | order_count
--------|------------
1       | 7
2       | 4
3       | 6
...
```

> **문제**: 원본 데이터(500건)가 사라짐. "각 고객의 첫 번째 주문"을 보고 싶다면?

### 윈도우 함수: 행은 유지하면서 계산

```sql
SELECT
    order_id,
    user_id,
    created_at,
    ROW_NUMBER() OVER (
        PARTITION BY user_id
        ORDER BY created_at
    ) as order_seq
FROM orders
LIMIT 20;
```

결과: **500행 유지** + 순번 추가

```
order_id | user_id | created_at          | order_seq
---------|---------|---------------------|----------
45       | 1       | 2025-11-03 10:00:00 | 1  ← 1번 고객의 첫 주문
123      | 1       | 2025-11-15 14:30:00 | 2  ← 1번 고객의 두 번째 주문
267      | 1       | 2025-12-01 19:00:00 | 3
89       | 2       | 2025-11-08 12:00:00 | 1  ← 2번 고객의 첫 주문
201      | 2       | 2025-11-22 18:30:00 | 2
...
```

### ROW_NUMBER 문법 분해

```
ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at)
    ↓                    ↓                    ↓
 순번 매기기         그룹 기준            정렬 기준
                 (user_id별로)      (날짜 오름차순)
```

| 요소 | 설명 |
|------|------|
| `ROW_NUMBER()` | 1부터 순번 부여 |
| `PARTITION BY` | 그룹 나누기 (GROUP BY와 비슷) |
| `ORDER BY` | 그룹 내 정렬 기준 |

### 핵심 패턴: 그룹별 첫 번째 찾기

**각 고객의 첫 주문만 조회** (DE 필수 패턴!)

```sql
WITH numbered AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY created_at
        ) as rn
    FROM orders
)
SELECT * FROM numbered
WHERE rn = 1
LIMIT 10;
```

**실무 활용**:
- **중복 제거**: 같은 user_id로 여러 번 들어온 데이터 중 최신 것만 남기기
- **첫 구매 분석**: 고객별 첫 구매 상품 파악
- **최근 데이터**: 각 센서의 가장 최근 측정값만 조회

### 응용: 카테고리별 최근 주문

```sql
-- 각 카테고리에서 가장 최근 주문 1건씩
WITH ranked AS (
    SELECT
        o.*,
        r.category,
        ROW_NUMBER() OVER (
            PARTITION BY r.category
            ORDER BY o.created_at DESC
        ) as rn
    FROM orders o
    JOIN restaurants r ON o.restaurant_id = r.restaurant_id
)
SELECT order_id, category, total_amount, created_at
FROM ranked
WHERE rn = 1;
```

결과: 카테고리당 1건씩 (치킨, 피자, 중식, 한식, 분식 = 5건)

---
## 2. CASE WHEN - 조건별 값 변환

### 기본 문법

```sql
SELECT
    order_id,
    total_amount,
    CASE
        WHEN total_amount >= 50000 THEN 'VIP'
        WHEN total_amount >= 30000 THEN '일반'
        ELSE '소액'
    END as order_grade
FROM orders
LIMIT 10;
```

결과:
```
order_id | total_amount | order_grade
---------|--------------|------------
1        | 23500        | 소액
2        | 45000        | 일반
3        | 52000        | VIP
...
```

### CASE WHEN 문법 구조

```
CASE
    WHEN 조건1 THEN 결과1
    WHEN 조건2 THEN 결과2
    ELSE 기본값
END as 새_컬럼명
```

> **주의**: 조건은 **위에서부터 순서대로** 평가됨 (먼저 만족하면 끝)

### 실무 활용: 상태값 정규화

원본 데이터가 지저분한 경우:
```
status 컬럼 값들: 'completed', 'COMPLETED', 'Complete', 'done', 'Done'...
```

**CASE WHEN으로 통일**:
```sql
SELECT
    order_id,
    status as original_status,
    CASE
        WHEN LOWER(status) IN ('completed', 'complete', 'done') THEN 'completed'
        WHEN LOWER(status) IN ('pending', 'wait', 'waiting') THEN 'pending'
        WHEN LOWER(status) IN ('cancelled', 'canceled', 'cancel') THEN 'cancelled'
        ELSE 'unknown'
    END as clean_status
FROM orders
LIMIT 10;
```

> **이게 바로 데이터 정제!** 지저분한 원본을 깨끗하게 변환

### CASE WHEN + 집계: 피벗 테이블

```sql
-- 지역별, 상태별 주문 수 (가로로 펼치기)
SELECT
    u.region,
    COUNT(CASE WHEN o.status = 'completed' THEN 1 END) as completed,
    COUNT(CASE WHEN o.status = 'pending' THEN 1 END) as pending,
    COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) as cancelled
FROM orders o
JOIN users u ON o.user_id = u.user_id
GROUP BY u.region;
```

결과:
```
region | completed | pending | cancelled
-------|-----------|---------|----------
강남   | 45        | 8       | 3
서초   | 38        | 12      | 5
...
```

---
## 3. COALESCE - NULL 처리

### NULL이란?
- "값이 없음"을 나타내는 특수한 상태
- 빈 문자열('')이나 0과 다름
- 계산하면 결과도 NULL (`NULL + 10 = NULL`)

### 🚨 의도적 오류: NULL 비교

```sql
-- ❌ 잘못된 방법 (결과: 0건)
SELECT * FROM users WHERE region = NULL;

-- ❌ 이것도 잘못됨 (결과: 0건)
SELECT * FROM users WHERE region != NULL;
```

**왜 안 될까?**
- NULL은 "모르는 값"이라서 **비교 자체가 불가능**
- `NULL = NULL`도 참이 아님!

```sql
-- ✅ 올바른 방법
SELECT * FROM users WHERE region IS NULL;
SELECT * FROM users WHERE region IS NOT NULL;
```

### COALESCE: NULL이면 기본값 사용

```sql
-- region이 NULL이면 '미지정'으로 표시
SELECT
    user_id,
    name,
    COALESCE(region, '미지정') as region
FROM users
LIMIT 10;
```

**여러 컬럼 중 첫 번째 NULL 아닌 값**:
```sql
-- phone이 NULL이면 email, 둘 다 NULL이면 '연락처없음'
SELECT
    user_id,
    COALESCE(phone, email, '연락처없음') as contact
FROM users;
```

### 실무 활용: JOIN 시 NULL 처리

```sql
-- 식당 정보가 없는 주문도 포함 (LEFT JOIN)
SELECT
    o.order_id,
    o.total_amount,
    COALESCE(r.name, '삭제된 식당') as restaurant_name,
    COALESCE(r.category, '미분류') as category
FROM orders o
LEFT JOIN restaurants r ON o.restaurant_id = r.restaurant_id
LIMIT 10;
```

> LEFT JOIN 결과에서 매칭 안 된 행은 NULL → COALESCE로 처리

---
## 4. 연습 문제

### 문제 1: ROW_NUMBER
각 카테고리별로 가장 최근 주문 1건씩만 조회하세요.

<details>
<summary>힌트</summary>

1. orders와 restaurants JOIN
2. ROW_NUMBER() OVER (PARTITION BY category ORDER BY created_at DESC)
3. WHERE rn = 1
</details>

<details>
<summary>정답</summary>

```sql
WITH ranked AS (
    SELECT
        o.*,
        r.category,
        ROW_NUMBER() OVER (
            PARTITION BY r.category
            ORDER BY o.created_at DESC
        ) as rn
    FROM orders o
    JOIN restaurants r ON o.restaurant_id = r.restaurant_id
)
SELECT * FROM ranked WHERE rn = 1;
```
</details>

### 문제 2: CASE WHEN
주문금액을 3단계로 분류하고, 단계별 주문 수를 집계하세요.
- 40,000원 이상: '대'
- 20,000원 이상: '중'
- 그 외: '소'

<details>
<summary>힌트</summary>

CASE WHEN으로 분류 → GROUP BY로 집계
</details>

<details>
<summary>정답</summary>

```sql
SELECT
    CASE
        WHEN total_amount >= 40000 THEN '대'
        WHEN total_amount >= 20000 THEN '중'
        ELSE '소'
    END as size_grade,
    COUNT(*) as order_count
FROM orders
GROUP BY
    CASE
        WHEN total_amount >= 40000 THEN '대'
        WHEN total_amount >= 20000 THEN '중'
        ELSE '소'
    END
ORDER BY order_count DESC;
```
</details>

### 문제 3 (보너스): COALESCE + JOIN
모든 주문에 대해 고객 지역을 표시하되, 지역이 NULL인 경우 '지역미상'으로 표시하세요.

<details>
<summary>힌트</summary>

orders와 users를 JOIN하고 COALESCE 적용
</details>

<details>
<summary>정답</summary>

```sql
SELECT
    o.order_id,
    o.total_amount,
    u.name as user_name,
    COALESCE(u.region, '지역미상') as region
FROM orders o
JOIN users u ON o.user_id = u.user_id
LIMIT 20;
```
</details>

---
## 5. 핵심 정리

| 함수 | 용도 | 예시 |
|------|------|------|
| ROW_NUMBER | 그룹별 순번 | `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` |
| CASE WHEN | 조건별 값 변환 | `CASE WHEN 조건 THEN 값 ELSE 기본값 END` |
| COALESCE | NULL 대체 | `COALESCE(컬럼, '기본값')` |

### 기억할 것
1. ROW_NUMBER + `WHERE rn = 1` = **그룹별 첫 번째/마지막 찾기**
2. CASE WHEN = **값 정규화, 분류**
3. NULL 비교는 `IS NULL` / `IS NOT NULL` 사용
4. COALESCE = **NULL을 기본값으로 대체**

---
## 6. 데이터 엔지니어와 데이터 정제

### 오늘 배운 함수들이 실무에서 쓰이는 곳

```
┌─────────────────────────────────────────────────────────────┐
│              데이터 정제 과정                                 │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   [원본 데이터]              [정제된 데이터]                  │
│   - 중복 있음          →     - 중복 제거 (ROW_NUMBER)        │
│   - 형식 제각각        →     - 형식 통일 (CASE WHEN)         │
│   - NULL 많음         →     - NULL 처리 (COALESCE)          │
│                                                             │
│   "있는 그대로"               "믿을 수 있는"                  │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

> **데이터 엔지니어의 핵심 업무 중 하나**:
> 지저분한 원본 데이터를 분석가가 바로 쓸 수 있게 정제하는 것

### 데이터 정제 패턴들

회사마다 데이터를 단계별로 정리하는 **다양한 패턴**이 있습니다:

| 패턴 | 특징 |
|------|------|
| 메달리온 아키텍처 | Databricks에서 대중화, Bronze/Silver/Gold 3단계 |
| Kimball 방법론 | 전통적인 데이터 웨어하우스 설계 |
| Data Vault | 이력 관리에 특화 |

> **지금 자세히 알 필요는 없습니다!**
>
> 중요한 건 **"데이터를 단계별로 정제한다"**는 개념이고,
> 오늘 배운 ROW_NUMBER, CASE WHEN, COALESCE가 그 과정에서 핵심적으로 사용됩니다.
>
> 나중에 Spark, Airflow를 배우면서 자연스럽게 다시 만나게 됩니다.

### 앞으로 배울 기술과의 연결

```
지금 (SQL)              →    나중에 (Spark, Airflow)
─────────────────────────────────────────────────────
수동으로 쿼리 실행       →    Airflow로 매일 자동 실행
500건 데이터            →    Spark로 수억 건 처리
PostgreSQL             →    데이터 레이크 (S3 등)

하지만 ROW_NUMBER, CASE WHEN, COALESCE는 똑같이 사용!
```

---

## 다음 시간 예고
DDL/DML 개념 + 기본키/외래키 + Docker 환경 구축 미션

---


# 3교시: DDL/DML과 데이터베이스 구조

## 학습 목표
- SQL 명령어의 4가지 분류 이해
- CREATE TABLE로 테이블이 만들어지는 과정 이해
- 기본키(PK)와 외래키(FK)의 역할 이해
- DE가 실무에서 어떤 SQL을 주로 사용하는지 파악

---

## 1. SQL 명령어 분류

"지금까지 SELECT만 했는데, 테이블은 어떻게 만들어졌을까?"

### SQL 명령어 4가지 분류

```
┌─────────────────────────────────────────────────────────────┐
│                    SQL 명령어 4가지 분류                      │
├─────────────────────────────────────────────────────────────┤
│  DDL (정의)       │ CREATE, ALTER, DROP, TRUNCATE           │
│  "테이블 만들기"   │ → 테이블 구조를 정의                      │
├─────────────────────────────────────────────────────────────┤
│  DML (조작)       │ SELECT, INSERT, UPDATE, DELETE          │
│  "데이터 다루기"   │ → 우리가 주로 하는 일 ★                  │
├─────────────────────────────────────────────────────────────┤
│  DCL (권한)       │ GRANT, REVOKE                           │
│  "누가 볼 수 있나" │ → DBA 업무, 알고만 있으면 됨              │
├─────────────────────────────────────────────────────────────┤
│  TCL (트랜잭션)   │ COMMIT, ROLLBACK                        │
│  "되돌리기"       │ → 운영 DB 작업 시 안전장치                │
└─────────────────────────────────────────────────────────────┘
```

### DE 관점에서의 중요도

| 분류 | 사용 빈도 | 언제 쓰나 |
|------|----------|----------|
| DDL | 가끔 | 파이프라인 초기 세팅, 새 테이블 생성 |
| DML | 매일 | SELECT 90%, INSERT 10% |
| DCL | 거의 안 씀 | DBA가 담당 |
| TCL | 가끔 | 운영 DB 직접 수정 시 |

---
## 2. DDL 살펴보기 - 우리 실습 환경이 만들어진 방법

### Day4 Docker 환경의 비밀

docker-compose.yml을 다시 보면:
```yaml
services:
  postgres:
    image: postgres:15
    volumes:
      - ./init:/docker-entrypoint-initdb.d  # ← 이 부분!
```

**`/docker-entrypoint-initdb.d`** 폴더 안의 `.sql` 파일들이 컨테이너 시작 시 **자동 실행**됩니다!

```
init/
├── 01_create_tables.sql   ← DDL (테이블 생성)
└── 02_seed_data.sql       ← DML (데이터 삽입)
```

### 실제 사용된 01_create_tables.sql

```sql
-- 테이블 생성 (DDL)
CREATE TABLE users (
    user_id SERIAL PRIMARY KEY,    -- 자동 증가 + 기본키
    name VARCHAR(50) NOT NULL,     -- 문자열, 필수
    region VARCHAR(20),            -- 문자열, 선택
    created_at TIMESTAMP DEFAULT NOW()  -- 기본값: 현재시간
);

CREATE TABLE restaurants (
    restaurant_id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    category VARCHAR(20),
    region VARCHAR(20)
);

CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    user_id INT REFERENCES users(user_id),        -- 외래키!
    restaurant_id INT REFERENCES restaurants(restaurant_id),  -- 외래키!
    total_amount INT,
    status VARCHAR(20) DEFAULT 'pending',
    created_at TIMESTAMP DEFAULT NOW()
);
```

> **컨테이너 올릴 때 이 파일이 실행되면서 테이블이 만들어진 거예요!**

### CREATE TABLE 각 요소 설명

| 요소 | 의미 | 예시 |
|------|------|------|
| `SERIAL` | 자동으로 1, 2, 3... 증가 | user_id가 자동 생성 |
| `PRIMARY KEY` | 기본키 (중복/NULL 불가) | 각 행의 고유 식별자 |
| `NOT NULL` | 필수 입력 | name은 반드시 있어야 함 |
| `DEFAULT` | 값 안 넣으면 이 값 사용 | created_at은 현재시간 |
| `VARCHAR(50)` | 최대 50자 문자열 | 이름은 50자까지 |
| `REFERENCES` | 외래키 설정 | 다른 테이블 참조 |

---
## 3. 기본키(PK)와 외래키(FK)

### 기본키(Primary Key)란?

각 행을 **유일하게 식별**하는 컬럼입니다.

마치 **주민등록번호**처럼 "이 사람이 누구인지" 구분하는 역할이에요.

- 중복 불가: 같은 값이 두 번 들어갈 수 없음
- NULL 불가: 반드시 값이 있어야 함
- 테이블당 1개만 설정 가능

### 외래키(Foreign Key)란?

orders 테이블에 주문을 넣는다고 생각해봅시다.

```sql
INSERT INTO orders (user_id, total_amount) VALUES (42, 25000);
```

여기서 **`user_id = 42`는 누구**일까요?

→ users 테이블에서 `user_id = 42`인 고객을 찾아야 합니다.

**그런데 만약 users 테이블에 42번 고객이 없다면?**

### FK가 없는 경우 - 문제 상황

```sql
-- orders 테이블에 FK가 없다고 가정
CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    user_id INT,  -- 그냥 INT, FK 아님
    total_amount INT
);

-- users에 42번 고객이 없는데...
INSERT INTO orders (user_id, total_amount) VALUES (42, 25000);
-- ✅ 그냥 들어감!
```

**나중에 문제 발생:**
```sql
-- "42번 고객 정보 보여줘"
SELECT * FROM users WHERE user_id = 42;
-- 😱 결과: 없음

-- JOIN하면?
SELECT o.*, u.name
FROM orders o
JOIN users u ON o.user_id = u.user_id
WHERE o.order_id = 100;
-- 😱 결과: 0건 (주문은 있는데 고객 정보가 없음!)
```

→ **"유령 주문"** 발생: 주문은 있는데 누가 했는지 모름

### FK가 있는 경우 - DB가 막아줌

```sql
-- orders 테이블 생성 시 FK 설정
CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    user_id INT REFERENCES users(user_id),  -- FK 설정!
    total_amount INT
);

-- users에 42번 고객이 없는데 주문 넣으려고 하면?
INSERT INTO orders (user_id, total_amount) VALUES (42, 25000);
```

**에러 발생:**
```
ERROR: insert or update on table "orders" violates foreign key constraint
DETAIL: Key (user_id)=(42) is not present in table "users".
```

→ **"users 테이블에 user_id=42가 없어서 넣을 수 없습니다"**

> DB가 미리 막아줌! 잘못된 데이터 자체가 안 들어감

### FK의 역할 정리

```
FK 없이 운영하면?
─────────────────────────────────
주문 데이터: user_id = 42, 9999, 12345...
고객 데이터: user_id = 1 ~ 100까지만 있음

→ "이 주문 누가 했어?" → "몰라, 없는 고객이야"
→ 분석할 때 매번 검증 쿼리 필요
→ 데이터 품질 문제 발생


FK 설정하면?
─────────────────────────────────
INSERT 할 때 user_id가 users에 있는지 자동 체크

→ 없으면 INSERT 자체가 안 됨
→ "모든 주문은 반드시 실제 고객과 연결됨" 보장
→ JOIN하면 항상 매칭됨
```

### 실무에서는 FK를 항상 쓸까?

**FK를 거는 경우:**
- 데이터 정합성이 매우 중요한 시스템
- 작은 규모의 서비스
- 실수로 잘못된 데이터 들어가면 큰일 나는 경우

**FK를 안 거는 경우도 많음:**
- 대용량 데이터 처리 시 (FK 체크도 시간이 걸림)
- 마이크로서비스 아키텍처 (DB가 분리됨)
- 대신 애플리케이션 레벨에서 검증

> **DE 입장에서는?**
> FK가 있든 없든 **데이터 정합성 확인은 필수!**
> FK가 없으면 직접 체크 쿼리를 작성해야 합니다.

---
## 4. DML 살펴보기 - 실제로 얼마나 쓰나?

### 실습 환경의 02_seed_data.sql

```sql
-- 데이터 삽입 (DML - INSERT)
INSERT INTO users (name, region) VALUES
    ('김민수', '강남'),
    ('이영희', '서초'),
    ('박철수', '송파');

INSERT INTO restaurants (name, category, region) VALUES
    ('맛있는치킨', '치킨', '강남'),
    ('피자천국', '피자', '서초');

INSERT INTO orders (user_id, restaurant_id, total_amount, status) VALUES
    (1, 1, 23000, 'completed'),
    (2, 2, 35000, 'completed');
```

> 컨테이너 시작 시 01_create_tables.sql 실행 후 이 파일이 실행됨

### DE가 DML을 직접 작성하는 경우

| 상황 | 빈도 | 설명 |
|------|------|------|
| SELECT | 매일 | 데이터 조회, 품질 확인, 분석 |
| INSERT (ETL) | 자주 | Python에서 DB로 데이터 적재 |
| INSERT (수동) | 가끔 | 테스트 데이터 추가 |
| UPDATE | 드물게 | 잘못된 데이터 보정 (주의!) |
| DELETE | 거의 안 함 | 운영 데이터 삭제는 위험 |

### 실무 현실

```python
# 대부분 이렇게 Python/Spark에서 처리
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('postgresql://...')
df.to_sql('table_name', engine, if_exists='append', index=False)

# INSERT문을 직접 작성하는 경우는 드뭄
# 주로 SELECT로 데이터 확인하는 일이 많음
```

> **DE는 SELECT 마스터가 되는 게 먼저!**
> INSERT/UPDATE는 도구(Python, Spark, Airflow)가 대신 해줍니다.

---
## 5. 연습 문제 (객관식)

### Q1. 다음 중 DDL(Data Definition Language)에 해당하는 명령어는?

- A) SELECT
- B) INSERT
- C) CREATE
- D) UPDATE

<details>
<summary>정답</summary>

**C) CREATE**

CREATE, ALTER, DROP, TRUNCATE가 DDL입니다.
테이블의 "구조"를 정의하는 명령어들이에요.
</details>

### Q2. PRIMARY KEY의 특징으로 올바른 것은?

- A) 중복 가능, NULL 가능
- B) 중복 불가, NULL 가능
- C) 중복 가능, NULL 불가
- D) 중복 불가, NULL 불가

<details>
<summary>정답</summary>

**D) 중복 불가, NULL 불가**

기본키는 각 행을 유일하게 식별해야 하므로,
같은 값이 두 번 있으면 안 되고 비어있어도 안 됩니다.
</details>

### Q3. 외래키(FK)를 설정했을 때 일어나는 일은?

- A) JOIN 속도가 빨라진다
- B) 참조하는 테이블에 없는 값을 넣으면 에러가 발생한다
- C) SELECT 할 때 자동으로 JOIN된다
- D) 컬럼 이름이 자동으로 같아진다

<details>
<summary>정답</summary>

**B) 참조하는 테이블에 없는 값을 넣으면 에러가 발생한다**

FK의 핵심 역할은 "데이터 무결성 보장"입니다.
users 테이블에 없는 user_id로 orders에 INSERT하면 DB가 막아줍니다.
</details>

### Q4. 데이터 엔지니어가 가장 자주 사용하는 DML 명령어는?

- A) INSERT
- B) UPDATE
- C) DELETE
- D) SELECT

<details>
<summary>정답</summary>

**D) SELECT**

DE 업무의 90%는 데이터 조회, 품질 확인, 분석입니다.
INSERT/UPDATE는 Python이나 Spark가 대신 처리하는 경우가 많습니다.
</details>

### Q5. Docker 컨테이너 시작 시 자동으로 SQL을 실행하려면 어디에 파일을 넣어야 하나?

- A) /var/lib/postgresql/
- B) /docker-entrypoint-initdb.d/
- C) /home/postgres/
- D) /etc/postgresql/

<details>
<summary>정답</summary>

**B) /docker-entrypoint-initdb.d/**

PostgreSQL 공식 이미지는 이 폴더의 .sql 파일을
컨테이너 최초 시작 시 알파벳 순서로 자동 실행합니다.
</details>

---
## 6. 핵심 정리

### SQL 명령어 분류
- **DDL**: 테이블 구조 정의 (CREATE, DROP)
- **DML**: 데이터 조작 (SELECT, INSERT, UPDATE, DELETE)
- **DCL/TCL**: 권한/트랜잭션 (알고만 있으면 됨)

### 기본키(PK)와 외래키(FK)
- **PK**: 각 행의 고유 식별자 (중복/NULL 불가)
- **FK**: 다른 테이블 참조, 데이터 무결성 보장
- FK가 없으면 "유령 데이터" 발생 가능

### DE 실무
- DDL은 초기 세팅 때 한 번
- DML 중 **SELECT가 90%**
- INSERT/UPDATE는 Python/Spark가 대신

---

## 다음 시간 예고
직접 docker-compose와 초기화 SQL을 작성해서 나만의 DB 환경 구축!

---


# 4-5교시: Docker 환경 구축 미션

## 학습 목표
- docker-compose.yml 직접 작성
- CREATE TABLE, INSERT SQL 파일 작성
- 컨테이너 실행하고 DB 환경 구축
- 오류 로그 읽고 문제 해결하는 방법 익히기

---

## 미션 개요

**상황**: 온라인 서점 "북마트"의 데이터베이스 환경을 구축합니다.

```
bookmart/
├── docker-compose.yml    ← 미션 1에서 작성
└── init/
    ├── 01_create_tables.sql  ← 미션 2에서 작성
    └── 02_seed_data.sql      ← 미션 3에서 작성
```

**북마트 스키마 (3개 테이블)**:
```
customers (고객 10명)
├── customer_id (PK)
├── name
└── email

books (책 20권)
├── book_id (PK)
├── title
├── price
└── category

orders (주문 50건)
├── order_id (PK)
├── customer_id (FK → customers)
├── book_id (FK → books)
├── quantity
└── order_date
```

---
## 미션 1: docker-compose.yml 작성 (25분)

### 폴더 생성

```bash
# 작업 폴더 생성
mkdir -p bookmart/init
cd bookmart
```

### docker-compose.yml 작성

아래 빈칸을 채워서 `docker-compose.yml` 파일을 작성하세요.

```yaml
version: '3.8'

services:
  postgres:
    image: postgres:15
    container_name: bookmart-db
    environment:
      POSTGRES_USER: bookmart
      POSTGRES_PASSWORD: ______      # 비밀번호 설정
      POSTGRES_DB: ______            # 데이터베이스 이름
    ports:
      - "______:5432"                # 호스트포트:컨테이너포트
    volumes:
      - ./init:/docker-entrypoint-initdb.d

  pgadmin:
    image: dpage/pgadmin4
    container_name: bookmart-pgadmin
    environment:
      PGADMIN_DEFAULT_EMAIL: admin@bookmart.com
      PGADMIN_DEFAULT_PASSWORD: admin
    ports:
      - "8080:80"
    depends_on:
      - postgres
```

<details>
<summary>정답</summary>

```yaml
version: '3.8'

services:
  postgres:
    image: postgres:15
    container_name: bookmart-db
    environment:
      POSTGRES_USER: bookmart
      POSTGRES_PASSWORD: bookmart123
      POSTGRES_DB: bookmart
    ports:
      - "5433:5432"    # 5432는 Day4에서 사용 중일 수 있어서 5433 사용
    volumes:
      - ./init:/docker-entrypoint-initdb.d

  pgadmin:
    image: dpage/pgadmin4
    container_name: bookmart-pgadmin
    environment:
      PGADMIN_DEFAULT_EMAIL: admin@bookmart.com
      PGADMIN_DEFAULT_PASSWORD: admin
    ports:
      - "8080:80"
    depends_on:
      - postgres
```
</details>

### 🚨 의도적 오류 1: 포트 충돌

만약 포트를 5432로 설정했는데 Day4 컨테이너가 아직 실행 중이라면?

```bash
docker compose up -d
```

**에러 로그:**
```
Error response from daemon: driver failed programming external connectivity
on endpoint bookmart-db: Bind for 0.0.0.0:5432 failed: port is already allocated
```

### 로그 읽는 법
```
"Bind for 0.0.0.0:5432 failed: port is already allocated"
         ↓
"5432 포트가 이미 사용 중이라서 연결할 수 없다"
```

### 해결 방법

**방법 1**: 기존 컨테이너 중지
```bash
# 실행 중인 컨테이너 확인
docker ps

# Day4 컨테이너 중지
docker stop [컨테이너_이름]
```

**방법 2**: 다른 포트 사용 (권장)
```yaml
ports:
  - "5433:5432"   # 호스트에서는 5433으로 접속
```

> **팁**: 여러 프로젝트를 동시에 띄울 때는 포트를 다르게 설정!

---
## 미션 2: CREATE TABLE 작성 (35분)

### init/01_create_tables.sql 작성

아래 빈칸을 채워서 테이블을 생성하세요.

```sql
-- 북마트 데이터베이스 테이블 생성

-- 1. 고객 테이블
CREATE TABLE customers (
    customer_id ______ PRIMARY KEY,   -- 자동 증가
    name VARCHAR(50) ______,          -- 필수 입력
    email VARCHAR(100)
);

-- 2. 책 테이블
CREATE TABLE books (
    book_id SERIAL PRIMARY KEY,
    title VARCHAR(200) NOT NULL,
    price ______,                     -- 정수형
    category VARCHAR(50)
);

-- 3. 주문 테이블
CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    customer_id INT ______ customers(customer_id),  -- 외래키
    book_id INT REFERENCES books(book_id),
    quantity INT DEFAULT 1,
    order_date TIMESTAMP DEFAULT NOW()
);
```

<details>
<summary>정답</summary>

```sql
-- 북마트 데이터베이스 테이블 생성

-- 1. 고객 테이블
CREATE TABLE customers (
    customer_id SERIAL PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    email VARCHAR(100)
);

-- 2. 책 테이블
CREATE TABLE books (
    book_id SERIAL PRIMARY KEY,
    title VARCHAR(200) NOT NULL,
    price INT,
    category VARCHAR(50)
);

-- 3. 주문 테이블
CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    customer_id INT REFERENCES customers(customer_id),
    book_id INT REFERENCES books(book_id),
    quantity INT DEFAULT 1,
    order_date TIMESTAMP DEFAULT NOW()
);
```
</details>

### 🚨 의도적 오류 2: FK 순서 문제

만약 테이블 생성 순서가 잘못되면?

```sql
-- ❌ 잘못된 순서
CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    customer_id INT REFERENCES customers(customer_id),  -- customers가 아직 없음!
    ...
);

CREATE TABLE customers (  -- 너무 늦게 생성
    customer_id SERIAL PRIMARY KEY,
    ...
);
```

**에러 로그 확인:**
```bash
docker compose logs postgres
```

**에러 메시지:**
```
ERROR: relation "customers" does not exist
```

### 로그 읽는 법
```
"relation "customers" does not exist"
          ↓
"customers 테이블이 존재하지 않는다"
          ↓
"orders 테이블이 customers를 참조하는데, customers가 아직 안 만들어짐"
```

### 해결 방법

**FK가 참조하는 테이블을 먼저 생성!**

```
올바른 순서:
1. customers (다른 테이블에 참조됨)
2. books (다른 테이블에 참조됨)
3. orders (customers, books를 참조함) ← 마지막에!
```

> **규칙**: 참조"되는" 테이블을 먼저, 참조"하는" 테이블을 나중에

---
## 미션 3: INSERT 데이터 작성 (25분)

### init/02_seed_data.sql 작성

```sql
-- 북마트 샘플 데이터 삽입

-- 1. 고객 데이터
INSERT INTO customers (name, email) VALUES
    ('홍길동', 'hong@email.com'),
    ('김영희', 'kim@email.com'),
    ('이철수', 'lee@email.com'),
    ('박지민', 'park@email.com'),
    ('최수진', 'choi@email.com');

-- 2. 책 데이터
INSERT INTO books (title, price, category) VALUES
    ('SQL 마스터', 35000, 'IT'),
    ('파이썬 입문', 28000, 'IT'),
    ('데이터 분석의 기술', 32000, 'IT'),
    ('클린 코드', 33000, 'IT'),
    ('경제학 원론', 25000, '경제'),
    ('마케팅 전략', 29000, '경영'),
    ('소설 모음집', 15000, '문학'),
    ('역사 이야기', 22000, '인문');

-- 3. 주문 데이터
INSERT INTO orders (customer_id, book_id, quantity) VALUES
    (1, 1, 2),   -- 홍길동이 SQL 마스터 2권
    (1, 2, 1),   -- 홍길동이 파이썬 입문 1권
    (2, 3, 1),   -- 김영희가 데이터 분석의 기술 1권
    (2, 1, 1),   -- 김영희가 SQL 마스터 1권
    (3, 4, 3),   -- 이철수가 클린 코드 3권
    (4, 5, 1),   -- 박지민이 경제학 원론 1권
    (5, 6, 2);   -- 최수진이 마케팅 전략 2권
```

### 🚨 의도적 오류 3: FK 제약 위반

만약 존재하지 않는 customer_id로 INSERT하면?

```sql
-- customers에 id=99인 고객이 없는데...
INSERT INTO orders (customer_id, book_id, quantity) VALUES (99, 1, 1);
```

**에러 로그 확인:**
```bash
docker compose logs postgres
```

**에러 메시지:**
```
ERROR: insert or update on table "orders" violates foreign key constraint "orders_customer_id_fkey"
DETAIL: Key (customer_id)=(99) is not present in table "customers".
```

### 로그 읽는 법
```
"violates foreign key constraint"
          ↓
"외래키 제약조건을 위반했다"

"Key (customer_id)=(99) is not present in table "customers""
          ↓
"customer_id=99가 customers 테이블에 없다"
```

### 해결 방법

**존재하는 customer_id만 사용!**

```sql
-- customers 테이블 확인
SELECT customer_id FROM customers;
-- 결과: 1, 2, 3, 4, 5

-- 존재하는 ID로 INSERT
INSERT INTO orders (customer_id, book_id, quantity) VALUES (1, 1, 1);
```

> **FK의 역할**: 잘못된 데이터가 들어가는 것을 DB가 막아줌!

---
## 미션 4: 환경 띄우고 확인 (35분)

### 컨테이너 실행

```bash
# bookmart 폴더에서 실행
cd bookmart

# 컨테이너 시작
docker compose up -d

# 로그 확인 (문제 없는지)
docker compose logs postgres
```

**정상 실행 시 로그:**
```
PostgreSQL init process complete; ready for start up.
LOG: database system is ready to accept connections
```

### VSCode SQLTools로 연결

**연결 정보:**
```
Host: localhost
Port: 5433        ← docker-compose.yml에서 설정한 포트
Database: bookmart
Username: bookmart
Password: bookmart123
```

### 데이터 확인 쿼리

```sql
-- 테이블 목록 확인
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';

-- 각 테이블 데이터 확인
SELECT * FROM customers;
SELECT * FROM books;
SELECT * FROM orders;
```

### JOIN으로 주문 상세 확인

```sql
-- 누가 무슨 책을 몇 권 주문했는지
SELECT
    c.name as 고객명,
    b.title as 책제목,
    b.price as 가격,
    o.quantity as 수량,
    b.price * o.quantity as 총액
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN books b ON o.book_id = b.book_id
ORDER BY c.name;
```

**결과 예시:**
```
고객명   | 책제목              | 가격  | 수량 | 총액
--------|--------------------| ------|-----|------
김영희  | SQL 마스터          | 35000 | 1   | 35000
김영희  | 데이터 분석의 기술   | 32000 | 1   | 32000
홍길동  | SQL 마스터          | 35000 | 2   | 70000
홍길동  | 파이썬 입문         | 28000 | 1   | 28000
...
```

> **성공!** 직접 만든 DB 환경에서 데이터 조회 완료!

---
## 트러블슈팅 가이드 정리

### 문제 해결 3단계

```
1단계: 로그 확인
─────────────────
docker compose logs postgres
docker compose logs [서비스명]

2단계: 에러 메시지 읽기
─────────────────
- "port is already allocated" → 포트 충돌
- "relation ... does not exist" → 테이블 순서 문제
- "violates foreign key constraint" → FK 위반

3단계: 수정 후 재시작
─────────────────
docker compose down        # 컨테이너 중지 및 삭제
# 파일 수정...
docker compose up -d       # 다시 시작
```

### 자주 쓰는 Docker 명령어

| 명령어 | 설명 |
|--------|------|
| `docker compose up -d` | 컨테이너 시작 (백그라운드) |
| `docker compose down` | 컨테이너 중지 및 삭제 |
| `docker compose logs [서비스]` | 로그 확인 |
| `docker compose ps` | 실행 중인 컨테이너 확인 |
| `docker ps` | 모든 실행 중인 컨테이너 |
| `docker exec -it [컨테이너] bash` | 컨테이너 내부 접속 |

---
## 핵심 정리

### 오늘 배운 것

1. **docker-compose.yml** 작성법
   - services, environment, ports, volumes

2. **초기화 SQL 파일**
   - `/docker-entrypoint-initdb.d/` 폴더에 넣으면 자동 실행
   - 파일 이름 순서대로 실행 (01_, 02_, ...)

3. **문제 해결 요령**
   - `docker compose logs`로 에러 확인
   - 에러 메시지 핵심 키워드 파악
   - 수정 후 `down` → `up` 재시작

### 기억할 것

- 포트 충돌 → 다른 포트 사용
- 테이블 순서 → 참조되는 테이블 먼저
- FK 에러 → 존재하는 값만 INSERT

---

## 다음 시간 예고
종합 실습: DE 인턴이 되어 실제 업무 요청 처리하기!

---


# 6-7교시: 종합 실습 - DE 인턴의 첫 주

## 학습 목표
- 지금까지 배운 SQL 총동원
- 실제 업무 요청을 시뮬레이션
- ETL 개념 자연스럽게 체험
- 이후 기술(Kafka, Spark, Airflow)과의 연결 이해

---

## 시나리오 설정

> 여러분은 온라인 서점 **"북마트"** 데이터팀 인턴입니다.
>
> 첫 주에 여러 부서에서 데이터 요청이 들어왔습니다.
> SQL로 해결해보세요!

**사용할 데이터**: 4-5교시에서 만든 북마트 DB
- customers (고객 5명)
- books (책 8권)
- orders (주문 7건)

---
## 미션 1: 데이터 추출 (Extract)

### 📩 마케팅팀 요청

> "이번 주 구매 고객 리스트 뽑아주세요.
> 고객 이름이랑 이메일이 필요해요."

### 힌트
- orders 테이블과 customers 테이블 JOIN
- 중복 고객 제거 필요 (같은 고객이 여러 번 주문했을 수 있음)

<details>
<summary>더 자세한 힌트</summary>

```sql
SELECT DISTINCT ...
FROM orders o
JOIN customers c ON ...
```
</details>

<details>
<summary>정답</summary>

```sql
SELECT DISTINCT
    c.name,
    c.email
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id;
```

결과:
```
name    | email
--------|------------------
홍길동  | hong@email.com
김영희  | kim@email.com
이철수  | lee@email.com
박지민  | park@email.com
최수진  | choi@email.com
```
</details>

---
## 미션 2: 데이터 변환 (Transform) - 고객 등급 분류

### 📩 분석팀 요청

> "고객별 총 구매액을 계산하고, 등급을 나눠주세요.
>
> - 10만원 이상: VIP
> - 5만원 이상: 일반
> - 그 외: 신규"

### 힌트
- 총 구매액 = price × quantity의 합계
- GROUP BY로 고객별 집계
- CASE WHEN으로 등급 분류

<details>
<summary>더 자세한 힌트</summary>

```sql
SELECT
    c.name,
    SUM(b.price * o.quantity) as total_amount,
    CASE
        WHEN SUM(...) >= 100000 THEN 'VIP'
        ...
    END as grade
FROM ...
GROUP BY ...
```
</details>

<details>
<summary>정답</summary>

```sql
SELECT
    c.name,
    c.email,
    SUM(b.price * o.quantity) as total_amount,
    CASE
        WHEN SUM(b.price * o.quantity) >= 100000 THEN 'VIP'
        WHEN SUM(b.price * o.quantity) >= 50000 THEN '일반'
        ELSE '신규'
    END as grade
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN books b ON o.book_id = b.book_id
GROUP BY c.customer_id, c.name, c.email
ORDER BY total_amount DESC;
```

결과:
```
name    | email            | total_amount | grade
--------|------------------|--------------|-------
이철수  | lee@email.com    | 99000        | 일반
홍길동  | hong@email.com   | 98000        | 일반
김영희  | kim@email.com    | 67000        | 일반
최수진  | choi@email.com   | 58000        | 일반
박지민  | park@email.com   | 25000        | 신규
```
</details>

---
## 미션 3: 데이터 정제 - 첫 구매 분석

### 📩 분석팀 추가 요청

> "각 고객의 첫 번째 구매가 어떤 책인지 알고 싶어요.
> 첫 구매 분석 리포트를 만들어주세요."

### 힌트
- ROW_NUMBER()로 고객별 주문 순서 매기기
- PARTITION BY customer_id, ORDER BY order_date
- WHERE rn = 1로 첫 번째만 필터

<details>
<summary>더 자세한 힌트</summary>

```sql
WITH numbered AS (
    SELECT
        ...,
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
        ) as rn
    FROM ...
)
SELECT ... FROM numbered WHERE rn = 1;
```
</details>

<details>
<summary>정답</summary>

```sql
WITH numbered AS (
    SELECT
        c.name as customer_name,
        b.title as book_title,
        b.category,
        o.order_date,
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_id
            ORDER BY o.order_date
        ) as rn
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN books b ON o.book_id = b.book_id
)
SELECT
    customer_name,
    book_title,
    category,
    order_date as first_order_date
FROM numbered
WHERE rn = 1;
```

결과:
```
customer_name | book_title         | category | first_order_date
--------------|--------------------| ---------|------------------
홍길동        | SQL 마스터          | IT       | 2025-01-09 ...
김영희        | 데이터 분석의 기술   | IT       | 2025-01-09 ...
이철수        | 클린 코드           | IT       | 2025-01-09 ...
박지민        | 경제학 원론         | 경제     | 2025-01-09 ...
최수진        | 마케팅 전략         | 경영     | 2025-01-09 ...
```
</details>

---
## 미션 4: 카테고리 분석

### 📩 경영지원팀 요청

> "카테고리별 매출 현황을 알려주세요.
> 어떤 카테고리가 가장 잘 팔리나요?"

### 힌트
- GROUP BY category
- 매출 = price × quantity 합계

<details>
<summary>정답</summary>

```sql
SELECT
    b.category,
    COUNT(*) as order_count,
    SUM(o.quantity) as total_quantity,
    SUM(b.price * o.quantity) as total_revenue
FROM orders o
JOIN books b ON o.book_id = b.book_id
GROUP BY b.category
ORDER BY total_revenue DESC;
```

결과:
```
category | order_count | total_quantity | total_revenue
---------|-------------|----------------|---------------
IT       | 5           | 8              | 289000
경영     | 1           | 2              | 58000
경제     | 1           | 1              | 25000
```
</details>

---
## 보너스 미션: 3테이블 JOIN + 복합 조건

### 📩 CEO 긴급 요청

> "IT 카테고리 책을 2권 이상 구매한 고객 중에서
> 총 구매액이 5만원 이상인 고객 명단이 필요합니다.
> 내일 이사회에서 발표할 자료예요."

### 힌트
- IT 카테고리 필터: WHERE category = 'IT'
- 2권 이상: HAVING SUM(quantity) >= 2
- 5만원 이상: HAVING SUM(price * quantity) >= 50000
- 두 조건 AND로 결합

<details>
<summary>정답</summary>

```sql
SELECT
    c.name,
    c.email,
    SUM(o.quantity) as it_book_count,
    SUM(b.price * o.quantity) as it_total_amount
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN books b ON o.book_id = b.book_id
WHERE b.category = 'IT'
GROUP BY c.customer_id, c.name, c.email
HAVING SUM(o.quantity) >= 2
   AND SUM(b.price * o.quantity) >= 50000
ORDER BY it_total_amount DESC;
```

결과:
```
name    | email            | it_book_count | it_total_amount
--------|------------------|---------------|----------------
이철수  | lee@email.com    | 3             | 99000
홍길동  | hong@email.com   | 3             | 98000
```
</details>

---
## ETL 개념 정리

### 오늘 우리가 한 일

```
┌─────────────────────────────────────────────────────────────┐
│                    ETL 파이프라인                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Extract (추출)                                             │
│  └─ SELECT, JOIN, WHERE로 필요한 데이터 뽑기                 │
│     → 미션 1: 구매 고객 리스트 추출                          │
│                                                             │
│  Transform (변환)                                           │
│  └─ CASE WHEN, ROW_NUMBER, GROUP BY로 가공                  │
│     → 미션 2: 고객 등급 분류                                 │
│     → 미션 3: 첫 구매 분석                                   │
│                                                             │
│  Load (적재)                                                │
│  └─ 결과를 새 테이블이나 파일로 저장                         │
│     → CREATE TABLE AS SELECT (CTAS)                        │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 참고: CTAS (Create Table As Select)

```sql
-- 미션 2 결과를 새 테이블로 저장
CREATE TABLE customer_grades AS
SELECT
    c.name,
    c.email,
    SUM(b.price * o.quantity) as total_amount,
    CASE
        WHEN SUM(b.price * o.quantity) >= 100000 THEN 'VIP'
        WHEN SUM(b.price * o.quantity) >= 50000 THEN '일반'
        ELSE '신규'
    END as grade
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN books b ON o.book_id = b.book_id
GROUP BY c.customer_id, c.name, c.email;

-- 이제 간단하게 조회 가능
SELECT * FROM customer_grades WHERE grade = 'VIP';
```

> 이렇게 가공된 테이블을 **"마트 테이블"** 또는 **"분석용 테이블"**이라고 부릅니다.

---
## 앞으로 배울 기술과의 연결

### 지금 vs 앞으로

```
┌─────────────────────────────────────────────────────────────┐
│           지금 (수동)              앞으로 (자동화)            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  SQL 직접 실행              Airflow                         │
│  → 매번 손으로 쿼리 실행     → 매일 새벽 자동 실행            │
│                             → 실패하면 알림                  │
│                                                             │
│  500건 데이터               Spark                           │
│  → 로컬 PostgreSQL          → 수억 건 분산 처리              │
│                             → 똑같은 SQL 문법 사용!          │
│                                                             │
│  배치로 한 번에             Kafka                           │
│  → 하루에 한 번 집계         → 실시간 스트리밍               │
│                             → 주문 들어올 때마다 처리        │
│                                                             │
│  Docker로 로컬 환경          AWS                            │
│  → 내 컴퓨터에서만           → 클라우드 인프라               │
│                             → 팀 전체가 공유                │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

> **핵심**: SQL 문법은 어디서나 똑같이 사용됩니다!
> SELECT, JOIN, GROUP BY, CASE WHEN, ROW_NUMBER...
> 오늘 배운 것들이 Spark SQL에서도 그대로 동작해요.

### 데이터 엔지니어의 업무 흐름

```
┌─────────────────────────────────────────────────────────────┐
│              DE가 하는 일 (큰 그림)                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 데이터 수집                                             │
│     └─ API, 크롤링, 로그 → Kafka로 수집                     │
│                                                             │
│  2. 데이터 저장                                             │
│     └─ DB, 데이터 레이크 (S3) → 오늘 배운 SQL로 조회        │
│                                                             │
│  3. 데이터 가공                                             │
│     └─ ETL 파이프라인 → Spark로 대용량 처리                 │
│                                                             │
│  4. 자동화                                                  │
│     └─ 스케줄링 → Airflow로 매일 자동 실행                  │
│                                                             │
│  5. 모니터링                                                │
│     └─ 데이터 품질 체크, 알림 설정                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

> 오늘은 2, 3번을 경험했습니다.
> 다음 주부터 1, 4, 5번을 배워갑니다!

---
## Day 5 전체 정리

### 오늘 배운 것

| 교시 | 내용 | 핵심 |
|------|------|------|
| 1교시 | 서브쿼리 | WHERE절, IN, CTE |
| 2교시 | 윈도우 함수 + 데이터 정제 | ROW_NUMBER, CASE WHEN, COALESCE |
| 3교시 | DDL/DML | CREATE TABLE, PK/FK, INSERT |
| 4-5교시 | Docker 환경 구축 | docker-compose, 초기화 SQL |
| 6-7교시 | 종합 실습 | ETL 체험, 실무 시뮬레이션 |

### SQL 실력 체크리스트

✅ SELECT, WHERE, ORDER BY, LIMIT
✅ GROUP BY, HAVING, 집계 함수
✅ JOIN (INNER, LEFT)
✅ 서브쿼리 (스칼라, IN, CTE)
✅ ROW_NUMBER
✅ CASE WHEN, COALESCE

> 이 정도면 **데이터 엔지니어 신입 면접 SQL**은 준비 완료!

---
## 다음 주 예고: Kafka

```
┌─────────────────────────────────────────────────────────────┐
│                    실시간 데이터 처리                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   지금: 배치 처리                                           │
│   "하루에 한 번 데이터 집계"                                 │
│                                                             │
│   다음 주: 스트림 처리                                       │
│   "주문이 들어올 때마다 실시간 처리"                         │
│                                                             │
│   ┌─────┐      ┌─────┐      ┌─────┐                        │
│   │ App │ ──▶  │Kafka│ ──▶  │ DB  │                        │
│   └─────┘      └─────┘      └─────┘                        │
│    주문         메시지        저장                           │
│    발생         큐           & 처리                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

> SQL은 계속 사용됩니다.
> Kafka로 들어온 데이터를 DB에 저장하고, SQL로 분석하는 흐름!

---

## 수고하셨습니다! 🎉